# Sessionization — the 30-Minute Gap Pattern

The single most-asked SQL pattern at Meta in 2026. Used in the screen, the onsite SQL round, the take-home, and the product-sense investigation round.

Pairs with `design/05_sessionization_pattern.md`.

In [1]:
import os, sqlite3
BASE = os.path.abspath(os.path.join(os.getcwd(), '..')) if 'notebooks' in os.getcwd() else os.getcwd()
schema_path = os.path.join(BASE, 'code', 'meta_schema.sql')
conn = sqlite3.connect(':memory:')
with open(schema_path) as f:
    conn.executescript(f.read())
print('schema loaded')

schema loaded


In [2]:
# Full sessionization query.
q = """
WITH events AS (
    SELECT user_id, event_ts,
           LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts) AS prev_ts
    FROM   instagram_story_events
),
gaps AS (
    SELECT user_id, event_ts, prev_ts,
           CASE WHEN prev_ts IS NULL
                 OR (JULIANDAY(event_ts) - JULIANDAY(prev_ts)) * 24 * 60 > 30
                THEN 1 ELSE 0
           END AS new_session
    FROM   events
),
session_ids AS (
    SELECT user_id, event_ts,
           SUM(new_session) OVER (PARTITION BY user_id ORDER BY event_ts) AS session_id
    FROM   gaps
)
SELECT user_id, session_id,
       MIN(event_ts) AS start_, MAX(event_ts) AS end_, COUNT(*) AS n
FROM   session_ids
GROUP BY user_id, session_id
ORDER BY user_id, session_id;
"""
for row in conn.execute(q):
    print(row)

(201, 1, '2026-01-01T08:00:00', '2026-01-01T08:05:00', 2)
(201, 2, '2026-01-01T08:45:00', '2026-01-01T08:45:00', 1)
(202, 1, '2026-01-01T09:00:00', '2026-01-01T09:00:00', 1)
(202, 2, '2026-01-02T09:00:00', '2026-01-02T09:00:00', 1)
(203, 1, '2026-01-01T10:00:00', '2026-01-01T10:00:00', 1)
(203, 2, '2026-01-02T10:00:00', '2026-01-02T10:00:00', 1)
(203, 3, '2026-01-03T10:00:00', '2026-01-03T10:00:00', 1)
(203, 4, '2026-01-04T10:00:00', '2026-01-04T10:00:00', 1)
(204, 1, '2026-01-01T12:00:00', '2026-01-01T12:00:00', 1)


## Sanity check: user 201 has 2 sessions

User 201's first 2 events (8:00, 8:05) are 5 min apart — same session. The third event (8:45) is 40 min after the second — new session. So 2 sessions for user 201.

## Reading the result table

- **User 201:** 2 sessions. The first contains both 8:00 and 8:05 events (5 min gap, under the 30-min threshold). The second is the 8:45 event — it starts a new session because the gap from 8:05 to 8:45 is 40 min, exceeding the 30-min session boundary. This is the **only** user in the sample data that demonstrates the sessionization logic; everyone else fires one event per day.
- **Users 202, 203, 204:** Each event is 24h+ after the previous one, so every event becomes its own session. User 203 fires on 4 consecutive days, producing 4 separate sessions — the canonical "one-off nightly" pattern that would normally be coalesced in a product dashboard.
- **What the test is asserting:** the `MIN/MAX/COUNT` window correctly partitions events into sessions per user, with the right session IDs. The crucial assertion is user 201's session_id = 2 — proving the cumulative-sum flag (the `SUM(new_session) OVER ...` line) properly increments when a new session starts.